# Basic RAG over a Serverless Codebase

A minimal RAG baseline using minsearch, following the course structure: `search`, `build_prompt`, `llm` and `rag`.

The data is the target repo for my project, [serverless-eda-insurance-claims-processing](https://github.com/aws-samples/serverless-eda-insurance-claims-processing), pinned to one commit. Run `make targets` first to clone it into `targets/`.

In [1]:
import json
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

import frontmatter
from gitsource import chunk_documents
from minsearch import Index
from openai import OpenAI

openai_client = OpenAI()

## 1. Load the Data

Two kinds of documents from the target repo:

- **docs**: READMEs and the EventCatalog pages. EventCatalog pages have frontmatter with the event's producers and consumers.
- **code**: CDK infrastructure and Lambda / agent code (`.ts`, `.py`, `.js` under `lib/` and `bin/`).

Having both lets the system answer from the code and point out where the docs disagree with it.

In [2]:
REPO_DIR = Path("../targets/insurance-claims")

SKIP_DIRS = {"node_modules", ".git", "cdk.out", ".github"}
DOC_EXTENSIONS = {".md", ".mdx"}
CODE_EXTENSIONS = {".ts", ".py", ".js"}


def is_skipped(path):
    return any(part in SKIP_DIRS for part in path.parts)


def as_text(value):
    if isinstance(value, list):
        return ", ".join(str(v) for v in value)
    return str(value) if value else ""


def load_doc(path):
    post = frontmatter.load(path)
    return {
        "kind": "docs",
        "filename": str(path.relative_to(REPO_DIR)),
        "title": as_text(post.metadata.get("name") or post.metadata.get("title") or path.parent.name),
        "summary": as_text(post.metadata.get("summary")),
        "producers": as_text(post.metadata.get("producers")),
        "consumers": as_text(post.metadata.get("consumers")),
        "content": post.content,
    }


def load_code(path):
    return {
        "kind": "code",
        "filename": str(path.relative_to(REPO_DIR)),
        "title": path.name,
        "summary": "",
        "producers": "",
        "consumers": "",
        "content": path.read_text(encoding="utf-8"),
    }


documents = []

for path in sorted(REPO_DIR.rglob("*")):
    if not path.is_file() or is_skipped(path.relative_to(REPO_DIR)):
        continue
    if path.suffix in DOC_EXTENSIONS:
        documents.append(load_doc(path))
    elif path.suffix in CODE_EXTENSIONS and path.parts[len(REPO_DIR.parts)] in {"lib", "bin"} and not path.name.endswith(".d.ts"):
        documents.append(load_code(path))

print(f"docs: {sum(d['kind'] == 'docs' for d in documents)}, code: {sum(d['kind'] == 'code' for d in documents)}")

docs: 43, code: 61


Chunk with a sliding window, as in the course. Each chunk also gets the line number it starts on, so answers can cite `file:line`.

In [3]:
contents = {d["filename"]: d["content"] for d in documents}

chunks = chunk_documents(documents, size=2000, step=1000)

for chunk in chunks:
    chunk["start_line"] = contents[chunk["filename"]][:chunk["start"]].count("\n") + 1

len(chunks)

355

## 2. Build the Index

In [4]:
index = Index(
    text_fields=["title", "summary", "producers", "consumers", "content", "filename"],
    keyword_fields=["kind", "filename"],
)

index.fit(chunks)

## 3. Define the RAG Functions

In [5]:
def search(query):
    return index.search(
        query,
        boost_dict={"title": 2.0, "producers": 1.5, "consumers": 1.5},
        num_results=8,
    )

In [6]:
instructions = """
You help engineers understand an unfamiliar AWS serverless codebase
(Lambda, EventBridge, SQS, Step Functions, DynamoDB, CDK) before they change it.

Answer the QUESTION using only the CONTEXT. The context has chunks of docs
and code, each with a filename and the line it starts on.

- Explain how the system works: which services produce and consume which
  events, and what each step does.
- Cite every claim as `filename:line`.
- Prefer the code over the docs. If the docs and the code disagree, say so
  and cite both.
- If the context doesn't contain the answer, say what's missing and where
  you would look next.
""".strip()


def build_prompt(query, search_results):
    context = [
        {
            "filename": r["filename"],
            "start_line": r["start_line"],
            "kind": r["kind"],
            "content": r["content"],
        }
        for r in search_results
    ]
    search_result_json = json.dumps(context, indent=2)

    user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

    return user_prompt

In [7]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [8]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## 4. Try It Out

In [9]:
print(rag("What happens after a claim is accepted? Which services consume the ClaimAccepted event and what do they do?"))

After a claim is accepted, the **Notifications Service** consumes the **ClaimAccepted** event. The purpose of this consumption is to inform the customer about the updated status of their claim. This is detailed in the documentation:

- "The Claims Service triggers ClaimAccepted event when a claim successfully passes validation and is accepted for further processing. The Notifications Service utilizes this event to inform the customer about the updated status of their claim." (event-catalog/events/ClaimAccepted/index.mdx:1)

This indicates a straightforward process where the claim acceptance triggers a notification to the corresponding customer. There are no conflicting statements in the provided context regarding what happens after a claim is accepted.
